# Word2vec

**Notebook 3 of 3.** Notebook 02 trained 2 numbers per word for 500 words, on a corpus small enough to watch every step. Good vectors need far more text than that — more than you will train on a laptop.

So this notebook trains no word vectors at all. It loads vectors somebody else trained on six billion words, asks what they know, does arithmetic with them — $\text{Berlin} - \text{Germany} + \text{France}$ — and then turns the same algorithm on something that is not language, where there is no pretrained model to borrow.

## Setup

In [ ]:
# standard library
import contextlib
import io
import time
from pathlib import Path

# data stack
import pandas as pd

# nlp stack
import gensim.downloader
from gensim.models import Word2Vec

## 1. From the Small Network to Word2vec

Notebook 02's network learned 500 words. For tens of thousands, and millions of pairs, it would be far too slow. **Word2vec** (Mikolov et al., 2013) does the same task — predict a target's context words, keep the hidden weights as vectors — with three steps made faster:

| | Notebook 02's network | Word2vec |
|---|---|---|
| Input | a one-hot vector, multiplied by the weights | the target's row, looked up directly |
| Output | a softmax over all 500 context words per pair | one yes/no question per pair: a real context word, or one of a few random ones? This is **negative sampling**, and only those few words are updated |
| Frequent words | removed with the stop-word list | most of their pairs skipped at random (**subsampling**) |

Fast enough, in other words, to run over billions of words — and somebody already has. **You will not train word vectors in this notebook.** Good ones need a corpus far larger than any you will hold on a laptop, and the people who own such corpora publish the result. This notebook uses theirs.

## 2. Pretrained Word Vectors

**GloVe** (Pennington et al., 2014) learns word vectors from the same idea as word2vec — which words appear near which — but counts co-occurrences over the whole corpus instead of sliding a window. `glove-wiki-gigaword-50` was trained on Wikipedia and a news corpus of six billion tokens, and holds 400,000 lowercase words with 50 numbers each.

[`gensim.downloader`](https://radimrehurek.com/gensim/downloader.html) fetches it. The first run downloads **66 MB** into `gensim-data` in your home directory; later runs read it from there.

In [ ]:
glove = gensim.downloader.load("glove-wiki-gigaword-50")
print(f"{len(glove.index_to_key):,} words with {glove.vector_size} numbers each")
print("Vector of 'sea':", glove["sea"][:6].round(3), "...", glove["sea"].shape)

Gensim's [`most_similar`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.most_similar) ranks the whole vocabulary by cosine similarity, exactly as your own `most_similar` did in notebook 02, and `similarity` scores a single pair. Both are rounded to six places below, because the differences that matter sit well past the second.

In [ ]:
def neighbors(word_vectors, word, topn=6):
    """Return the topn words most similar to word, with their similarity rounded to six places."""
    pairs = []
    for neighbor, similarity in word_vectors.most_similar(word, topn=topn):
        pairs.append(f"{neighbor} {similarity:.6f}")
    return pairs


for word in ["sea", "night", "soul", "bee"]:
    print(f"{word:6s}", ", ".join(neighbors(glove, word, topn=4)))

**Result.** These are the neighbors of Wikipedia and the news, not of poetry. `sea` gives `ocean`, `waters`, `seas`, `coast`. `night` gives `day` first of all — opposites keep the same company, as they did in notebook 02.

Two are a warning. `soul` gives `love`, then `pop`, `album`, `band`: in a news corpus, soul is mostly a kind of music. `bee` gives `gees`, because *Bee Gees* appears far more often than the insect. **The vectors know the senses of their training texts and no others**, which is why six billion words of news is not automatically better than your own corpus — it is better at news.

In [ ]:
for first, second in [("sea", "ocean"), ("sea", "bee"), ("day", "night"), ("summer", "winter")]:
    print(f"similarity({first}, {second}) = {glove.similarity(first, second):.6f}")

**Result.** `sea` and `ocean` score 0.881165, `sea` and `bee` 0.225608: the measure separates related from unrelated cleanly.

Then look at the opposites. `day` and `night` score 0.874673 and `summer` and `winter` **0.919984** — higher than `sea` and `ocean`. Antonyms are the hardest case for an embedding, because they appear in exactly the same places (*by day and by night*, *summer and winter alike*). Similar **use** is not similar **meaning**, and nothing in the method can tell the two apart.

## 3. Arithmetic on Words

The **directions** between vectors carry meaning of their own. The step from Germany to Berlin — the vector $\text{Berlin} - \text{Germany}$, "the capital of" — points roughly the same way as the step from France to Paris. Add that step to France and you land near Paris:

$$\text{Berlin} - \text{Germany} + \text{France} \approx \text{Paris}$$

Note the order: it is the capital minus **its own** country, plus the new one. The other way round, $\text{Germany} - \text{Berlin} + \text{France}$, gives Belgium — a country near Germany, which is what you asked for without meaning to.

GloVe's vocabulary is lowercase throughout, so the cell below asks for `berlin` and `france` even though this paragraph spells them as the names they are.

In gensim the words to add go in `positive` and the words to subtract in `negative`. `most_similar` leaves out the words you passed in; otherwise `france` itself would come first.

In [ ]:
ANALOGIES = [
    (["berlin", "france"], ["germany"]),   # a capital and its country
    (["french", "germany"], ["france"]),   # a language and its country
    (["cats", "dog"], ["cat"]),            # a singular and its plural
    (["hotter", "cold"], ["hot"]),         # a comparative
]

for positive, negative in ANALOGIES:
    answer = glove.most_similar(positive=positive, negative=negative, topn=3)
    print(f"{positive[0]} - {negative[0]} + {positive[1]} =",
          ", ".join(f"{word} {score:.6f}" for word, score in answer))

**Result.** All four land:

| | | |
| :-- | :-- | --: |
| $\text{Berlin} - \text{Germany} + \text{France}$ | `paris` | 0.916864 |
| $\text{French} - \text{France} + \text{Germany}$ | `german` | 0.974839 |
| $\text{cats} - \text{cat} + \text{dog}$ | `dogs` | 0.904021 |
| $\text{hotter} - \text{hot} + \text{cold}$ | `colder` | 0.825890 |

One direction means *the capital of*, another *the language of*, another *the plural of*, another *the comparative of*. Nobody put them there; they fall out of which words keep company with which.

Grammar is the surest, 0.974839 and 0.904021, because every text repeats it. Geography is looser, and the runner-up says why: second behind `paris` comes `prohertrib`, which is not a word at all but a scanning error common enough in six billion words to have earned a vector of its own.

**Your turn:** find an analogy of your own. A verb form ($\text{walking} - \text{walked} + \text{swam}$), another capital, or a relation you invent. Not every one works, and the failures are as interesting as the successes — $\text{Asia} - \text{Thailand} + \text{Congo}$ ought to give Africa, and does not.

In [ ]:
# TODO: the words to add and to subtract, then pass them to
# glove.most_similar(positive=..., negative=..., topn=3)
# your code:


## 4. The Same Algorithm Without Any Words

Nothing in word2vec knows what a word is. It reads a sequence of symbols and learns a vector for each from the symbols around it, so give it sequences that are not language and it does the same thing.

`data/baskets.csv` holds 33,653 shopping baskets of a British online gift shop, one row each, as the stock codes bought together. A basket is a sentence; a product code is a word. `data/products.csv` says what each code is, and the model never sees it.

In [ ]:
baskets = pd.read_csv(Path("data") / "baskets.csv")
products = pd.read_csv(Path("data") / "products.csv").set_index("code")["description"]
print(f"{len(baskets):,} baskets, {len(products):,} products")
baskets.head(3)

In [ ]:
# One basket per row becomes one list of codes per row, the shape gensim expects
shopping = baskets["items"].str.split().tolist()
print("The first basket:", shopping[0])
print("As products:     ", [products[code] for code in shopping[0][:4]], "...")

No pretrained model exists for a gift shop's stock codes, so this is the one model the notebook trains. `window=10` reaches most of a basket, whose median size is 16, so order inside it hardly matters; `vector_size=32` is enough for 4,000 products; five passes suffice for 628,000 items. `train_word2vec` fixes the seed and uses one worker so the result is reproducible, and hides a harmless message gensim prints while training.

In [ ]:
def train_word2vec(sentences, **parameters):
    """Train a reproducible Word2Vec model (seed 42, one worker) on a list of token lists."""
    with contextlib.redirect_stderr(io.StringIO()):  # hide the harmless gensim 4.4 message
        return Word2Vec(sentences=sentences, seed=42, workers=1, **parameters)

In [ ]:
start = time.perf_counter()
shop = train_word2vec(shopping, vector_size=32, window=10, min_count=5, sg=1, negative=10, epochs=5)
print(f"Trained in {time.perf_counter() - start:.0f} s, {len(shop.wv.index_to_key):,} products have a vector")

In [ ]:
def recommend(code, topn=5):
    """Return the products whose vectors are closest to `code`, with their descriptions."""
    rows = []
    for other, similarity in shop.wv.most_similar(code, topn=topn):
        rows.append({"code": other, "description": products[other], "similarity": round(similarity, 6)})
    return pd.DataFrame(rows).set_index("code")


print("Bought with:", products["22423"])
recommend("22423")

**Result.** `REGENCY CAKESTAND 3 TIER` brings back the Roses, Green and Pink Regency teacups, the teapot and the tea plate, 0.89 to 0.93.

Nothing told the model that these are crockery or that `22699` relates to `22423`. It saw the codes share baskets — the evidence it used for `sea` and `ocean`, with the words taken away. Notebook 01 gave every word a column of its own, so `sea` and `ocean` were as unrelated as `sea` and `bee`; a vector learned from company puts them together, and the algorithm never needed them to be words.

**Your turn:** pick another product and look at what comes with it. `products.sample(10)` shows ten codes to choose from, or search the descriptions with `products[products.str.contains("LUNCH BOX")]`.

In [ ]:
# TODO: choose a code from products and pass it to recommend()
# your code:


## Summary

- Word2vec does notebook 02's task, made fast by row lookups, negative sampling and subsampling — fast enough that the vectors worth having have already been trained by somebody else.
- `gensim.downloader.load` fetches them, `most_similar` ranks a word's neighbors and `similarity` scores a pair. Words **used** alike come out close, including opposites such as `day` and `night`.
- The **directions** between vectors carry relations, so $\text{Berlin} - \text{Germany} + \text{France}$ lands on `paris`. The order matters, and not every relation is there to be found.
- The algorithm never needed words. Where no pretrained model exists — a shop's stock codes — train one on whatever sequences you have, and it learns what goes together.

## Sources

- Loading pretrained vectors and running word2vec on things that are not words both come from *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst (O'Reilly, 2024), chapter 2. The book's own example recommends songs from playlists; this notebook uses shopping baskets, because that data can be redistributed.
- The basket data is the **Online Retail II** dataset of the UCI Machine Learning Repository (Chen, 2019), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). `data/build_baskets.py` shows how it was cut down.
- `glove-wiki-gigaword-50`: Pennington, J., Socher, R. and Manning, C. D. (2014), [GloVe: Global Vectors for Word Representation](https://nlp.stanford.edu/projects/glove/).

The repository [README](README.md) lists every source and licence.